# Dataset Cleaning


## 1. Setup & Configuration


In [2]:
import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime
import re
import warnings

warnings.filterwarnings("ignore")

In [12]:
# --- CONFIGURATION ---
DATA_DIR = Path("../data/aftenposten")
OUTPUT_DIR = Path("../data/output")
OUTPUT_DIR.mkdir(exist_ok=True)

# Treatment date: Stoltenberg appointed finance minister
TREATMENT_DATE = pd.Timestamp(
    "2025-02-05"
)  # Date of appointment (Feb 4, 2025) so it'll be in the news on Feb 5, 2025

# Analysis windows (4 weeks before and after)
BEFORE_START = pd.Timestamp("2025-01-06")
BEFORE_END = pd.Timestamp("2025-02-04")  # day before appointment
AFTER_START = pd.Timestamp("2025-02-06")  # day after appointment
AFTER_END = pd.Timestamp("2025-03-05")
ELECTION_START = pd.Timestamp("2025-08-06")  # Start of election campaign period
ELECTION_END = pd.Timestamp("2025-09-10")  # Election day

# Valid values for validation
VALID_ISSUE_CODES = {"IMM", "ECON", "SEC", "GOV", "TAX", "GAZ", "CLI", "HEA", "OTH"}
VALID_YN = {"YES", "NO", ""}
VALID_TONE = {"POS", "NEU", "NEG", ""}
VALID_CONCERNS = {"YES", "NO"}

# Issue groupings for hypothesis tests
ISSUE_GROUPS = {
    "FrP-owned": ["IMM", "ECON"],  # Immigration + cost-of-living
    "Labour-owned": ["SEC", "GOV"],  # Security + governance
    "Contested": ["TAX"],  # Wealth tax
    "Left-niche": ["CLI", "GAZ", "HEA"],  # Climate, Gaza, health
    "Other": ["OTH"],
}

# Reverse mapping: code -> group name
CODE_TO_GROUP = {}
for group, codes in ISSUE_GROUPS.items():
    for code in codes:
        CODE_TO_GROUP[code] = group

print(f"Data directory: {DATA_DIR}")
print(f"Treatment date: {TREATMENT_DATE.date()}")
print(f"BEFORE period: {BEFORE_START.date()} to {BEFORE_END.date()}")
print(f"AFTER period:  {AFTER_START.date()} to {AFTER_END.date()}")
print(f"ELECTION period: {ELECTION_START.date()} to {ELECTION_END.date()}")


Data directory: ../data/aftenposten
Treatment date: 2025-02-05
BEFORE period: 2025-01-06 to 2025-02-04
AFTER period:  2025-02-06 to 2025-03-05
ELECTION period: 2025-08-06 to 2025-09-10


## 2. Parse raw text files


In [5]:
def parse_article_block(block_text: str, source_file: str) -> dict | None:
    """Parse a single article block (one article's fields) into a dictionary."""
    lines = block_text.strip().split("\n")
    article = {"_source_file": source_file}

    for line in lines:
        line = line.strip()
        if not line:
            continue
        # Split on first colon only (values may contain colons)
        match = re.match(r"^([A-Z_]+(?:_\d+)?)\s*:\s*(.*)", line)
        if match:
            key = match.group(1).strip()
            value = match.group(2).strip()
            article[key] = value
        else:
            # Continuation line — append to last key
            if article:
                last_key = list(article.keys())[-1]
                if last_key != "_source_file":
                    article[last_key] += " " + line

    # Minimum required fields check
    required = {"HEADLINE_NO", "ISSUE_CODE", "PUBLICATION_DATE"}
    if not required.issubset(article.keys()):
        missing = required - set(article.keys())
        print(f"  ⚠ Skipping incomplete block in {source_file}: missing {missing}")
        return None

    return article


def parse_file(filepath: Path) -> list[dict]:
    """Parse a single .txt file containing multiple article blocks."""
    text = filepath.read_text(encoding="utf-8")

    # Split on double newlines (blank lines between blocks)
    # Handle both \n\n and \r\n\r\n
    blocks = re.split(r"\n\s*\n", text)

    articles = []
    for block in blocks:
        block = block.strip()
        if not block:
            continue
        article = parse_article_block(block, filepath.name)
        if article:
            articles.append(article)

    return articles


# Parse all files
all_articles = []
txt_files = sorted(DATA_DIR.glob("*.txt"))

if not txt_files:
    print(f"⚠ No .txt files found in {DATA_DIR}/")
    print("  Make sure your data files are in the data/aftenposten/ directory.")
else:
    print(f"Found {len(txt_files)} files to parse:\n")
    for f in txt_files:
        articles = parse_file(f)
        print(f"  {f.name}: {len(articles)} articles")
        all_articles.extend(articles)

    print(f"\nTotal articles parsed: {len(all_articles)}")


Found 54 files to parse:

  2025-01-06.txt: 10 articles
  2025-01-13.txt: 14 articles
  2025-01-20.txt: 11 articles
  2025-01-21.txt: 14 articles
  2025-01-22.txt: 17 articles
  2025-01-23.txt: 17 articles
  2025-01-24.txt: 17 articles
  2025-01-25.txt: 20 articles
  2025-01-26.txt: 12 articles
  2025-01-27.txt: 13 articles
  2025-01-28.txt: 18 articles
  2025-01-29.txt: 16 articles
  2025-01-30.txt: 18 articles
  2025-01-31.txt: 20 articles
  2025-02-01.txt: 19 articles
  2025-02-02.txt: 12 articles
  2025-02-03.txt: 10 articles
  2025-02-04.txt: 16 articles
  2025-02-05.txt: 14 articles
  2025-02-06.txt: 18 articles
  2025-02-07.txt: 15 articles
  2025-02-08.txt: 17 articles
  2025-02-09.txt: 9 articles
  2025-02-10.txt: 8 articles
  2025-02-11.txt: 12 articles
  2025-02-12.txt: 14 articles
  2025-02-13.txt: 15 articles
  2025-02-14.txt: 17 articles
  2025-02-15.txt: 18 articles
  2025-02-16.txt: 14 articles
  2025-02-17.txt: 14 articles
  2025-02-18.txt: 16 articles
  2025-02-19.txt

## 3. Build DataFrame & clean


In [6]:
df_raw = pd.DataFrame(all_articles)

if df_raw.empty:
    print("No data parsed. Check your data/aftenposten/ directory.")
else:
    print(f"Raw DataFrame: {len(df_raw)} rows, {len(df_raw.columns)} columns")
    print(f"Columns: {list(df_raw.columns)}")

# %%
# Standardize column names & types
COLUMN_RENAME = {
    "NEWSPAPER": "newspaper",
    "CONCERNS_NORWAY": "concerns_norway",
    "PUBLICATION_DATE": "date",
    "SECTION_NO": "section_no",
    "HEADLINE_NO": "headline_no",
    "HEADLINE_EN": "headline_en",
    "LEAD_EN": "lead_en",
    "ISSUE_CODE": "issue_code",
    "ISSUE_CODE_2": "issue_code_2",
    "MENTIONS_STOLTENBERG": "mentions_stoltenberg",
    "MENTIONS_LISTHAUG": "mentions_listhaug",
    "MENTIONS_SOLBERG": "mentions_solberg",
    "TONE_AP": "tone_ap",
    "BRIEF_JUSTIFICATION": "justification",
    "_source_file": "source_file",
}

df = df_raw.rename(columns=COLUMN_RENAME)

# Keep only known columns (ignore any extras)
known_cols = list(COLUMN_RENAME.values())
existing_cols = [c for c in known_cols if c in df.columns]
extra_cols = [c for c in df.columns if c not in known_cols]
if extra_cols:
    print(f"Note: dropping unexpected columns: {extra_cols}")
df = df[existing_cols].copy()

# Parse date
df["date"] = pd.to_datetime(df["date"], errors="coerce")

# Uppercase and strip string fields
str_cols = [
    "issue_code",
    "issue_code_2",
    "concerns_norway",
    "mentions_stoltenberg",
    "mentions_listhaug",
    "mentions_solberg",
    "tone_ap",
]
for col in str_cols:
    if col in df.columns:
        df[col] = df[col].astype(str).str.strip().str.upper()
        df[col] = df[col].replace({"NAN": "", "NONE": "", "": ""})

print(f"Cleaned DataFrame: {len(df)} rows")
df.head(3)

Raw DataFrame: 821 rows, 15 columns
Columns: ['_source_file', 'NEWSPAPER', 'CONCERNS_NORWAY', 'PUBLICATION_DATE', 'SECTION_NO', 'HEADLINE_NO', 'HEADLINE_EN', 'LEAD_EN', 'ISSUE_CODE', 'ISSUE_CODE_2', 'MENTIONS_STOLTENBERG', 'MENTIONS_LISTHAUG', 'MENTIONS_SOLBERG', 'TONE_AP', 'BRIEF_JUSTIFICATION']
Cleaned DataFrame: 821 rows


,newspaper,concerns_norway,date,section_no,headline_no,headline_en,lead_en,issue_code,issue_code_2,mentions_stoltenberg,mentions_listhaug,mentions_solberg,tone_ap,justification,source_file
0,Aftenposten,YES,2025-01-06,Ideer & Samfunn,"2025 er altså ikke Assads år, men det kan bli ...","2025 is not Assad’s year, but it could be Syria’s",The columnist reflects on the fall of Assad’s ...,SEC,HEA,NO,NO,NO,NEU,"The article is primarily about Syria’s war, re...",2025-01-06.txt
1,Aftenposten,YES,2025-01-06,Nyheter | Nyheter Arbeidsliv,Tonje Brenna vil øke aldersgrensen i hele arbe...,Tonje Brenna wants to raise the age limit acro...,Labour Minister Tonje Brenna wants the upper a...,HEA,GOV,NO,NO,NO,POS,"The article is mainly about pensions, working ...",2025-01-06.txt
2,Aftenposten,YES,2025-01-06,Nyheter | Stortingsvalget 2025,Kaller utviklingen tilhøyre og Ap for oppsikts...,Calls the development for the Conservatives an...,A new polling average shows historically weak ...,GOV,,NO,NO,YES,NEG,"The article is mainly about party support, pol...",2025-01-06.txt


## 4. Validate


In [7]:
issues = []

# Check dates
bad_dates = df["date"].isna().sum()
if bad_dates:
    issues.append(f"  {bad_dates} rows have unparseable dates")

# Check issue codes
if "issue_code" in df.columns:
    invalid_codes = df[~df["issue_code"].isin(VALID_ISSUE_CODES)]
    if len(invalid_codes):
        issues.append(
            f"  {len(invalid_codes)} rows have invalid issue_code: "
            f"{invalid_codes['issue_code'].unique()}"
        )

# Check issue_code_2 (allow blank)
if "issue_code_2" in df.columns:
    valid_code2 = VALID_ISSUE_CODES | {""}
    invalid_c2 = df[~df["issue_code_2"].isin(valid_code2)]
    if len(invalid_c2):
        issues.append(
            f"  {len(invalid_c2)} rows have invalid issue_code_2: "
            f"{invalid_c2['issue_code_2'].unique()}"
        )

# Check yes/no fields
for col in [
    "mentions_stoltenberg",
    "mentions_listhaug",
    "mentions_solberg",
    "concerns_norway",
]:
    if col in df.columns:
        invalid = df[~df[col].isin(VALID_YN | VALID_CONCERNS)]
        if len(invalid):
            issues.append(
                f"  {len(invalid)} rows have invalid {col}: {invalid[col].unique()}"
            )

# Check tone
if "tone_ap" in df.columns:
    invalid_tone = df[~df["tone_ap"].isin(VALID_TONE)]
    if len(invalid_tone):
        issues.append(
            f"  {len(invalid_tone)} rows have invalid tone_ap: "
            f"{invalid_tone['tone_ap'].unique()}"
        )

if issues:
    print("⚠ VALIDATION ISSUES FOUND:")
    for i in issues:
        print(i)
    print("\nFix these in your source .txt files and re-run.")
else:
    print("✓ All validation checks passed.")

✓ All validation checks passed.


## 5. Add analytical columns


In [13]:
# --- Filter: only articles that concern Norway ---
""" if "concerns_norway" in df.columns:
    n_before_filter = len(df)
    df_norway = df[df["concerns_norway"] == "YES"].copy()
    n_dropped = n_before_filter - len(df_norway)
    print(
        f"Filtered to Norway-related articles: {len(df_norway)} "
        f"(dropped {n_dropped} non-Norway articles)"
    )
else:
    df_norway = df.copy()
    print("No 'concerns_norway' column found — keeping all articles.") """

df_norway = df.copy()


# --- Assign period ---
def assign_period(date):
    if pd.isna(date):
        return "UNKNOWN"
    if BEFORE_START <= date <= BEFORE_END:
        return "BEFORE"
    elif AFTER_START <= date <= AFTER_END:
        return "AFTER"
    elif ELECTION_START <= date <= ELECTION_END:
        return "ELECTION"
    elif date == TREATMENT_DATE:
        return "TREATMENT_DAY"
    else:
        return "OUTSIDE"


df_norway["period"] = df_norway["date"].apply(assign_period)

# --- Assign issue group ---
df_norway["issue_group"] = df_norway["issue_code"].map(CODE_TO_GROUP).fillna("Other")

# --- Boolean columns for easier analysis ---
for col in ["mentions_stoltenberg", "mentions_listhaug", "mentions_solberg"]:
    if col in df_norway.columns:
        bool_col = col.replace("mentions_", "flag_")
        df_norway[bool_col] = df_norway[col] == "YES"

# --- Week number (for time-series plots) ---
df_norway["week"] = df_norway["date"].dt.isocalendar().week.astype(int)
df_norway["year_week"] = (
    df_norway["date"].dt.year.astype(str)
    + "-W"
    + df_norway["date"].dt.isocalendar().week.astype(str).str.zfill(2)
)

# --- Day relative to treatment ---
df_norway["days_from_treatment"] = (df_norway["date"] - TREATMENT_DATE).dt.days

print(f"\nPeriod distribution:")
print(df_norway["period"].value_counts().to_string())

# Show the working dataset
print(f"\nWorking dataset: {len(df_norway)} articles")
df_norway.head(3)



Period distribution:
period
ELECTION         298
BEFORE           274
AFTER            235
TREATMENT_DAY     14

Working dataset: 821 articles


,newspaper,concerns_norway,date,section_no,headline_no,headline_en,lead_en,issue_code,issue_code_2,mentions_stoltenberg,...,justification,source_file,period,issue_group,flag_stoltenberg,flag_listhaug,flag_solberg,week,year_week,days_from_treatment
0,Aftenposten,YES,2025-01-06,Ideer & Samfunn,"2025 er altså ikke Assads år, men det kan bli ...","2025 is not Assad’s year, but it could be Syria’s",The columnist reflects on the fall of Assad’s ...,SEC,HEA,NO,...,"The article is primarily about Syria’s war, re...",2025-01-06.txt,BEFORE,Labour-owned,False,False,False,2,2025-W02,-30
1,Aftenposten,YES,2025-01-06,Nyheter | Nyheter Arbeidsliv,Tonje Brenna vil øke aldersgrensen i hele arbe...,Tonje Brenna wants to raise the age limit acro...,Labour Minister Tonje Brenna wants the upper a...,HEA,GOV,NO,...,"The article is mainly about pensions, working ...",2025-01-06.txt,BEFORE,Left-niche,False,False,False,2,2025-W02,-30
2,Aftenposten,YES,2025-01-06,Nyheter | Stortingsvalget 2025,Kaller utviklingen tilhøyre og Ap for oppsikts...,Calls the development for the Conservatives an...,A new polling average shows historically weak ...,GOV,,NO,...,"The article is mainly about party support, pol...",2025-01-06.txt,BEFORE,Labour-owned,False,False,True,2,2025-W02,-30


## 6. Build the analysis sample


In [14]:
df_analysis = df_norway[
    df_norway["period"].isin(["BEFORE", "AFTER", "ELECTION", "TREATMENT_DAY"])
].copy()
print(f"Analysis sample: {len(df_analysis)} articles")
print(f"  BEFORE: {(df_analysis['period'] == 'BEFORE').sum()}")
print(f"  AFTER:  {(df_analysis['period'] == 'AFTER').sum()}")
print(f"  ELECTION: {(df_analysis['period'] == 'ELECTION').sum()}")
print(f"  TREATMENT_DAY: {(df_analysis['period'] == 'TREATMENT_DAY').sum()}")

Analysis sample: 821 articles
  BEFORE: 274
  AFTER:  235
  ELECTION: 298
  TREATMENT_DAY: 14


## 7. Save outputs


In [16]:
# Save the full cleaned dataset
df_norway.to_csv(OUTPUT_DIR / "aftenposten_all_articles.csv", index=False)
df_analysis.to_csv(OUTPUT_DIR / "aftenposten_analysis_sample.csv", index=False)

print(f"Saved to {OUTPUT_DIR}/:")
print(f"  aftenposten_all_articles.csv     ({len(df_norway)} rows)")
print(f"  aftenposten_analysis_sample.csv  ({len(df_analysis)} rows)")


Saved to ../data/output/:
  aftenposten_all_articles.csv     (821 rows)
  aftenposten_analysis_sample.csv  (821 rows)
